[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/1RuwxBMQiWajzAo2wvU-L_prc9Hw_UZip/view?usp=drive_link)

# LLM Evaluation – Full Dataset

This notebook demonstrates how to evaluate LLM responses when you already have question–answer pairs. Each sample includes `user_input` and `llm_response`; Floeval runs metrics directly without generating responses.

**Objectives**
- Install Floeval and configure credentials
- Load a full dataset from a JSON file you provide
- Configure the LLM provider and evaluation metrics
- Run the evaluation and inspect aggregate and per-sample results

## 1. Installation

Install Floeval before running this notebook.

In [ ]:
%pip install floeval>=0.2.0b1

## 2. Configuration Constants

Set the following constants before running. Replace placeholder values with your API credentials and model identifiers.

**Provider flexibility:** You can use any OpenAI-compatible provider (OpenAI, Azure OpenAI, Anthropic, local models, etc.) — set the appropriate `base_url` and model names for your provider.

**Using FloTorch:** If you want to use FloTorch keys and gateway, obtain credentials from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

In [ ]:
import getpass
# LLM and API configuration

OPENAI_BASE_URL = "https://api.openai.com/v1"
OPENAI_API_KEY = getpass.getpass("your-api-key")
OPENAI_CHAT_MODEL = "gpt-4o-mini"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

## 3. Imports

Import evaluation components and the LLM configuration schema. RAGAS metrics require both chat and embedding models.

In [ ]:
from pathlib import Path

from floeval import Evaluation, DatasetLoader
from floeval.config.schemas.io.llm import OpenAIProviderConfig

## 4. Configure the LLM

Build `OpenAIProviderConfig` for evaluation metrics. RAGAS `answer_relevancy` uses both chat and embedding models.

In [ ]:
llm_config = OpenAIProviderConfig(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    chat_model=OPENAI_CHAT_MODEL,
    embedding_model=OPENAI_EMBEDDING_MODEL,
)

## 5. Load the Dataset

Minimal JSON shape:

```json
{
  "samples": [
    { "user_input": "...", "llm_response": "..."}
  ]
}
```

**Example file** — full dataset with `user_input`, `llm_response`, optional `ground_truth`.  
<a href="../datasets/llm_evaluation/sample_llm_full_dataset.json" download="sample_llm_full_dataset.json">sample_llm_full_dataset.json</a>

Provide the dataset JSON path (or upload in Colab), then load it with `DatasetLoader`.


In [ ]:
try:
    from google.colab import files
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    print("Upload your dataset JSON file:")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    dataset_path = Path(next(iter(uploaded.keys())))
else:
    dataset_path = Path(input("Enter path to dataset JSON file: ").strip().strip('"')).expanduser()


### Resolve Dataset Path

Provide `dataset_path` via file upload in Colab or local JSON path input in Jupyter.


In [ ]:
dataset = DatasetLoader.from_file(dataset_path, partial_dataset=False)
print(f"Dataset loaded from {dataset_path}: {len(dataset.samples)} samples")


### Load Full Dataset

Load and validate the full dataset with `DatasetLoader.from_file(..., partial_dataset=False)`.


## 6. Create and Run the Evaluation

Instantiate `Evaluation` with the dataset, provider config, and `answer_relevancy`, then run scoring.

In [ ]:
evaluation = Evaluation(
    dataset=dataset,
    llm_config=llm_config,
    metrics=["answer_relevancy"],
    default_provider="ragas",
    metric_params={"answer_relevancy": {"threshold": 0.8}},
)


### Build Evaluation Object

Configure `Evaluation` with dataset, `llm_config`, and metric settings.


In [ ]:
results = evaluation.run()
print("Aggregate scores:", results.aggregate_scores)


### Run Evaluation

Execute `evaluation.run()` to compute and print aggregate metric scores.


## 7. Inspect Per-Sample Results

Each row in `results.sample_results` contains metric outputs (score, pass/fail, and metadata), enabling detailed sample-level inspection.

### Inspect per-sample results

Iterates `results.sample_results` to print each question snippet and metric scores.


In [ ]:
for i, sr in enumerate(results.sample_results, start=1):
    print(f"Sample {i}: {sr['user_input'][:50]}...")
    for key, data in sr.get("metrics", {}).items():
        print(f"  {key}: score={data.get('score')}, passed={data.get('passed')}")

## Summary

This notebook demonstrated the end-to-end process of evaluating LLM responses using a full dataset with Floeval.

The key components included:

1. **Dataset Loading**: A full dataset was loaded from JSON using `DatasetLoader.from_file`.
2. **LLM Configuration**: The OpenAI-compatible provider was configured with chat and embedding models for RAGAS metrics.
3. **Evaluation Execution**: The `answer_relevancy` metric was run via the RAGAS provider to score answer quality.
4. **Results Inspection**: Aggregate scores and per-sample metrics were accessed through `results.aggregate_scores` and `results.sample_results`.

This example showcases the standard workflow for evaluating pre-generated LLM outputs with Floeval.